In [1]:
from pathlib import Path
import os, sys

PROJECT_ROOT = Path.cwd().resolve().parents[1]
if Path.cwd().resolve() != PROJECT_ROOT:
    os.chdir(PROJECT_ROOT)

root_str = str(PROJECT_ROOT)
if root_str in sys.path:
    sys.path.remove(root_str)
sys.path.insert(0, root_str)

#print("PROJECT_ROOT =", PROJECT_ROOT)
#print("cwd =", Path.cwd())

In [2]:
%load_ext autoreload
%autoreload 2

In [3]:
import scanpy as sc
import pandas as pd
import numpy as np
import os
import torch
from datetime import datetime
import yaml

In [4]:
control_key = "is_control"
condition_rep_keys = "perturbation_embeddings"
condition_combined_keys = "condition_combined"
mass_deduct_keys = "bc1_well"
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

condition_keys = "cytokine"
data_origin = "PBMC_all_hvg2000_gemini_test1_42_cellflow_0_True_X_pca_100_None"  #{dataset_name}_{random_seed}_{test_ratio}_{zero_shot}_{sample_rep_scaled}_{n_comps}
sample_rep = "X_pca"

data_origin_ot = "PBMC_all_hvg2000_42_cellflow_0_True_X_pca_100_None"

In [5]:
preprocess_save_path = f"./data/processed/{data_origin}"
adata_control = sc.read_h5ad(f"{preprocess_save_path}_control.h5ad",backed='r')
adata_train = sc.read_h5ad(f"{preprocess_save_path}_train.h5ad",backed='r')
if os.path.exists(f"{preprocess_save_path}_test.h5ad"):
    adata_test = sc.read_h5ad(f"{preprocess_save_path}_test.h5ad",backed='r')
else:
    adata_test = None
print(adata_control)
print(adata_train)
print(adata_test)

AnnData object with n_obs × n_vars = 629701 × 2000 backed at 'data/processed/PBMC_all_hvg2000_gemini_test1_42_cellflow_0_True_X_pca_100_None_control.h5ad'
    obs: 'sample', 'species', 'gene_count', 'tscp_count', 'mread_count', 'bc1_wind', 'bc2_wind', 'bc3_wind', 'bc1_well', 'bc2_well', 'bc3_well', 'log1p_n_genes_by_counts', 'log1p_total_counts', 'total_counts_MT', 'pct_counts_MT', 'log1p_total_counts_MT', 'donor', 'cytokine', 'treatment', 'cell_type', 'donor_one_hot', 'is_control', 'donor_idx', 'condition_combined'
    var: 'n_cells', 'highly_variable', 'means', 'dispersions', 'dispersions_norm'
    uns: 'cov_config', 'global_rulebook', 'hvg', 'log1p', 'normalized_m', 'pca', 'perturbation_embeddings'
    obsm: 'X_pca'
    varm: 'PCs', 'X_mean'
AnnData object with n_obs × n_vars = 7863725 × 2000 backed at 'data/processed/PBMC_all_hvg2000_gemini_test1_42_cellflow_0_True_X_pca_100_None_train.h5ad'
    obs: 'sample', 'species', 'gene_count', 'tscp_count', 'mread_count', 'bc1_wind', 'bc2_w

In [7]:
#adata_control.uns

## training

In [8]:
from src.training import pre_compute_wfr_ot, seed_everything
from src.training import FNet,train
from torch.optim.lr_scheduler import CosineAnnealingLR

In [ ]:
delta = 8
reg_m = 2
use_mini_batch_uot = True
group_number = 5
ot_origin = f"delta{delta}_regm{reg_m}"
save_path_train = f"./experiments/ot_plan/ot_precompute_{data_origin_ot}_{ot_origin}_train.pkl"
save_path_test = f"./experiments/ot_plan/ot_precompute_{data_origin_ot}_{ot_origin}_test.pkl"

load_saved = True
if os.path.exists(save_path_train) and load_saved:
    print(f"read{save_path_train}")
    ot_results_train = pd.read_pickle(save_path_train)
else:
    ot_results_train = pre_compute_wfr_ot(adata_control, 
                                          adata_train, 
                                          condition_keys=condition_keys,
                                          condition_combined_keys=condition_combined_keys,
                                          save_path = save_path_train, 
                                          sample_rep=sample_rep, 
                                          mass_deduct_keys=mass_deduct_keys,
                                          delta = delta,
                                          reg_m = reg_m,
                                          group_number = group_number,
                                          use_mini_batch_uot = use_mini_batch_uot,
                                          batch_save_size=50,
                                          draw = False)

if os.path.exists(save_path_test) and load_saved:
    print(f"read{save_path_test}")
    ot_results_test = pd.read_pickle(save_path_test)
else:
    ot_results_test = pre_compute_wfr_ot(adata_control, 
                                         adata_test, 
                                         condition_keys=condition_keys,
                                         condition_combined_keys=condition_combined_keys,
                                         save_path = save_path_test, 
                                         sample_rep=sample_rep, 
                                         mass_deduct_keys=mass_deduct_keys,
                                         delta = delta,
                                         reg_m = reg_m,
                                         group_number = group_number,
                                         use_mini_batch_uot = use_mini_batch_uot,
                                         batch_save_size=50,
                                         draw=False)


/lustre/home/2300012136/mycode/DynamicVirtualCell/WFRFM/src/training/utils.py:214: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  control_gb = adata_control.obs.groupby(control_groups_keys)
/lustre/home/2300012136/mycode/DynamicVirtualCell/WFRFM/src/training/utils.py:218: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  treated_gb = adata_treated.obs.groupby(treated_groupby_keys)
100%|██████████| 100/100 [31:10<00:00, 18.70s/it]


Batch saved: 100/936


100%|██████████| 100/100 [30:15<00:00, 18.15s/it]


Batch saved: 200/936


100%|██████████| 100/100 [28:06<00:00, 16.86s/it]


Batch saved: 300/936


  0%|          | 0/100 [00:00<?, ?it/s]

In [42]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

n_iterations = 10000

load_experiment = False # 如果是False可以无视下面三个参数
experiment_load_path = "experiments/exp_20260224_155124_PBMC_donor11_hvg_42_0.4_False_X_pca_scaled_100_False_delta10_regm1"
config_load_path = os.path.join(experiment_load_path, "config.yaml")
model_load_path = os.path.join(experiment_load_path, "checkpoints", "test_11_epoch_20000.pt") 


if not load_experiment:
    experiment_save_path = os.path.join("experiments", f"exp_{timestamp}_{data_origin}_{ot_origin}")
    checkpoint_save_path = os.path.join(experiment_save_path, "checkpoints")
    os.makedirs(checkpoint_save_path, exist_ok=True)
    
    config = {
        "in_out_dim": adata_control.obsm[sample_rep].shape[1],
        "condition_dim": next(iter(adata_train.uns[condition_rep_keys].values())).shape[0],
        
        "hidden_dim_v": 2048,   
        "hidden_dim_g": 128,
        "n_hiddens_v": 8,
        "n_hiddens_g": 6,
    
        "con_embedding_dim": 256,
        "hidden_dim_con": 512,
        
        "time_dim": 1024,
        "time_embedding_dim": 128,
        "hidden_dim_time": 512,

        "cov_emb_dim": 128, 
        
        "bottle_dim": 256,
        "activation": 'SiLU',
        
        "batch_size_per_condition": 512,  
        "batch_size_condition":30,
        "learning_rate": 1e-4,  
        "dropout": 0.1,
        "weight_decay":1e-4,
        "eta_min":5e-5,
        "seed": 42,
        "eval_interval":10,
    }
    config_save_path = f"{experiment_save_path}/config.yaml"
    with open(config_save_path, 'w') as f:
        yaml.dump(config, f, default_flow_style=False, sort_keys=False)
        print(f"config saved to {config_save_path}")
else:
    with open(config_load_path, 'r') as f:
        config = yaml.safe_load(f)
        print(f"load saved_config from {config_load_path}")

    experiment_save_path = experiment_load_path  #直接继续某一个实验
    checkpoint_save_path = os.path.join(experiment_save_path, "checkpoints")


model = FNet(
    in_out_dim=config["in_out_dim"], 
    rulebook = adata_control.uns["global_rulebook"],
    hidden_dim_v=config["hidden_dim_v"], 
    n_hiddens_v=config["n_hiddens_v"], 
    hidden_dim_g=config["hidden_dim_g"], 
    n_hiddens_g=config["n_hiddens_g"], 
    condition_dim=config["condition_dim"], 
    con_embedding_dim=config["con_embedding_dim"], 
    hidden_dim_con=config["hidden_dim_con"], 
    time_dim=config["time_dim"], 
    time_embedding_dim=config["time_embedding_dim"], 
    hidden_dim_time=config["hidden_dim_time"], 
    bottle_dim=config["bottle_dim"], 
    cov_emb_dim=config.get("cov_emb_dim", 64), 
    dropout=config["dropout"], 
    activation=config["activation"]
)

model = model.to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=config["learning_rate"], weight_decay=config["weight_decay"])
scheduler = CosineAnnealingLR(optimizer, T_max=n_iterations, eta_min=config["eta_min"])
seed_everything(config["seed"])

if load_experiment:    
    if os.path.exists(model_load_path):
        checkpoint = torch.load(model_load_path, map_location=device)
        model.load_state_dict(checkpoint['model_state_dict'])
        optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
        scheduler.load_state_dict(checkpoint['scheduler_state_dict'])
        print(f"load model from {model_load_path}")
    else:
        print("wrong model_load_path")

print("setup complete")

cuda
config saved to experiments/exp_20260405_142759_PBMC_all_hvg2000_gemini_test1_42_cellflow_0_True_X_pca_100_None_delta8_regm2/config.yaml
Global seed set to 42
setup complete


In [ ]:
model_save_path = os.path.join(checkpoint_save_path, "test_11")
train.train_model(adata_control, adata_train, adata_test,
                ot_results_train, ot_results_test, 
                model.to(device),
                optimizer,
                scheduler = scheduler,
                n_iterations= n_iterations,
                batch_size_per_condition = config["batch_size_per_condition"],
                batch_size_condition = config["batch_size_condition"],
                sample_rep = sample_rep,
                condition_rep_keys = condition_rep_keys,
                device = device,
                save_path=model_save_path,
                eval_interval = config["eval_interval"],
                save_interval = 5000, #每隔多少轮保存一次
                save_only_last = True) # 是否只保留最后的pt文件


正在根据 Rulebook 构建协变量路由表...
正在构建局部到全局的 OT 索引映射...
DataLoaderHelper 构建完成！
正在根据 Rulebook 构建协变量路由表...
正在构建局部到全局的 OT 索引映射...
DataLoaderHelper 构建完成！


Begin flow and growth matching...:   0%|          | 22/10000 [00:23<3:03:12,  1.10s/epoch, loss=0.916, vloss=0.759, gloss=0.157, test_loss=1.085, test_vloss=0.849, test_gloss=0.236]

In [ ]:
model_save_path

In [ ]:
model_save_path = "experiments/exp_20260219_213059_PBMC_donor11_hvg_42_0.4_False_X_pca_scaled_100_delta10_regm1/checkpoints/test"

In [ ]:
from src.evaluate import draw_loss
load_path = f"{model_save_path}_epoch_20000.pt"
draw_loss(load_path,eval_interval = 10,cut = 100,save_path = None) # cut代表去掉前多少个epoch 避免前期loss过大影响观察